# Tema: SCD Type 2 e intervalos de validez

## Objetivos
Construir historia por cliente con intervalos semiabiertos y consultar el estado en una fecha.

## Conceptos importantes para el examen
SCD2 conserva versiones; valid_from inclusivo y valid_to exclusivo; una fila vigente por clave; secuencia de negocio; AUTO CDC SEQUENCE BY para pipelines.

**Dificultad:** Examen · **Tiempo estimado:** 90 min.



Ejecuta la preparación una vez; después avanza celda a celda. Las soluciones modifican datos: úsalas tras tu intento. Para volver al estado inicial, ejecuta de nuevo la preparación completa (crea otro schema). No uses «Run all» para estudiar.

- [ ] Completado
- [ ] Necesito repasar
- [ ] Dominado

## Preparación y datos ficticios
Se necesita un notebook Python en Databricks con Spark y Unity Catalog. Solo se crean objetos en el schema de prácticas mostrado.

In [ ]:
# Cada ejecución de esta celda crea un schema NUEVO y aislado.
# El catálogo debe existir y permitir USE CATALOG y CREATE SCHEMA.
# Si no puedes crear schemas, pide uno de prácticas exclusivo y cambia SCHEMA.
import re
import uuid
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.window import Window

dbutils.widgets.text("catalog", spark.sql("SELECT current_catalog()").first()[0])
CATALOG = dbutils.widgets.get("catalog")
RUN_ID = uuid.uuid4().hex[:10]
SCHEMA = "dea_21_" + RUN_ID
def ident(value):
    return "`" + value.replace("`", "``") + "`"
spark.sql(f"USE CATALOG {ident(CATALOG)}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {ident(SCHEMA)}")
spark.sql(f"USE SCHEMA {ident(SCHEMA)}")
spark.sql("SET TIME ZONE 'UTC'")
print(f"Objetos de esta sesión: {CATALOG}.{SCHEMA}")
# No se borran automáticamente schemas, tablas ni checkpoints.


In [ ]:
customers = spark.createDataFrame(
    [(i, f"Cliente {i:02d}", ["Madrid", "Sevilla", "Bilbao"][i % 3],
      datetime(2026, 1, 1)) for i in range(1, 13)],
    "customer_id INT, name STRING, city STRING, updated_at TIMESTAMP"
)
customers.write.format("delta").mode("overwrite").saveAsTable("customers")
updates = spark.createDataFrame([
    (2, "Cliente 02", "Valencia", datetime(2026, 2, 1)),
    (5, "Cliente 05", "Zaragoza", datetime(2026, 2, 2)),
    (13, "Cliente 13", "Madrid", datetime(2026, 2, 3))],
    customers.schema)
updates.write.format("delta").mode("overwrite").saveAsTable("customers_updates")
display(customers)
# Historial de eventos del laboratorio: se conserva completo para reconstrucción batch.
history_events = customers.unionByName(updates).unionByName(updates.filter("customer_id=2").withColumn("city",F.lit("Lugo")).withColumn("updated_at",F.to_timestamp(F.lit("2026-03-01"))))
history_events.write.format("delta").mode("overwrite").saveAsTable("customer_events")

## PARTE 1 - EJEMPLOS GUIADOS

### 1. Construir intervalos desde historial completo
Recomputación batch didáctica, válida para estos datos pequeños. No es un algoritmo incremental de producción.

In [ ]:
def build_scd2():
    src = spark.table("customer_events").dropDuplicates(["customer_id","updated_at"])
    w = Window.partitionBy("customer_id").orderBy("updated_at")
    return (src.withColumn("valid_from",F.col("updated_at"))
        .withColumn("valid_to",F.lead("updated_at").over(w))
        .withColumn("is_current",F.col("valid_to").isNull()))
build_scd2().write.format("delta").mode("overwrite").saveAsTable("dim_customer_history")
display(spark.table("dim_customer_history").filter("customer_id=2").orderBy("valid_from"))

### 2. Consulta temporal

In [ ]:
%sql
SELECT customer_id, name, city FROM dim_customer_history
WHERE valid_from <= TIMESTAMP '2026-02-15'
  AND (valid_to > TIMESTAMP '2026-02-15' OR valid_to IS NULL);

### 3. Invariantes
Esta fuente no tiene conflictos con misma clave y fecha. En un origen real añade event_seq; no uses dropDuplicates para elegir arbitrariamente entre valores distintos.

In [ ]:
history = spark.table("dim_customer_history")
assert history.filter("is_current").groupBy("customer_id").count().filter("count != 1").count() == 0
assert history.filter("valid_to IS NOT NULL AND valid_to <= valid_from").count() == 0

## PARTE 2 - EJERCICIOS
Resuelve todos antes de abrir las soluciones. Los ejercicios se realizan en orden y pueden usar resultados anteriores.

### EJERCICIO 1
Cuenta versiones del cliente 2 y filas actuales totales.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 2
Consulta ciudad del 2 el 15 de enero, 15 de febrero y 15 de marzo.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 3
Introduce un evento tardío del 2 el 15 de febrero en Cádiz y reconstruye. Verifica que divide el intervalo existente.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 4
Comprueba ausencia de solapamientos y continuidad entre versiones contiguas.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 5
Une tres compras ficticias del cliente 2 con la ciudad válida en cada fecha, manteniendo una fila por compra.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


## PARTE 3 - PISTAS
**Pista 1:** Tres estados del 2; 13 claves actuales.

**Pista 2:** Aplica el intervalo [valid_from, valid_to).

**Pista 3:** La fecha de negocio determina el orden, no la llegada.

**Pista 4:** lead(valid_from) debería coincidir con valid_to.

**Pista 5:** Join por clave y condiciones de intervalo.

## PARTE 4 - SOLUCIONES
**Detente aquí si todavía estás practicando.** Referencias completas para comparar después de resolver. Puedes plegar esta sección en Databricks.

### Solución 1

In [ ]:
assert spark.table("dim_customer_history").filter("customer_id=2").count() == 3
assert spark.table("dim_customer_history").filter("is_current").count() == 13

### Solución 2

In [ ]:
for date in ["2026-01-15","2026-02-15","2026-03-15"]:
    display(spark.sql(f"""SELECT '{date}' AS as_of, city FROM dim_customer_history WHERE customer_id=2
    AND valid_from <= TIMESTAMP '{date}' AND (valid_to > TIMESTAMP '{date}' OR valid_to IS NULL)"""))

### Solución 3

In [ ]:
spark.createDataFrame([(2,"Cliente 02","Cádiz",datetime(2026,2,15))],customers.schema).write.format("delta").mode("append").saveAsTable("customer_events")
build_scd2().write.format("delta").mode("overwrite").saveAsTable("dim_customer_history")
assert spark.table("dim_customer_history").filter("customer_id=2").count() == 4
display(spark.table("dim_customer_history").filter("customer_id=2").orderBy("valid_from"))

### Solución 4

In [ ]:
w = Window.partitionBy("customer_id").orderBy("valid_from")
check = spark.table("dim_customer_history").withColumn("next_start",F.lead("valid_from").over(w))
assert check.filter("next_start IS NOT NULL AND NOT (valid_to <=> next_start)").count() == 0

### Solución 5

In [ ]:
orders = spark.createDataFrame([(1,2,datetime(2026,1,20)),(2,2,datetime(2026,2,20)),(3,2,datetime(2026,3,20))], "order_id INT, customer_id INT, order_time TIMESTAMP")
o,h = orders.alias("o"), spark.table("dim_customer_history").alias("h")
result = o.join(h,(F.col("o.customer_id")==F.col("h.customer_id")) & (F.col("o.order_time")>=F.col("h.valid_from")) & (F.col("h.valid_to").isNull() | (F.col("o.order_time")<F.col("h.valid_to"))),"left").select("o.order_id","h.city")
assert result.count() == 3
display(result)

## PARTE 5 - PREGUNTAS TIPO EXAMEN
Preguntas originales de práctica; no son preguntas oficiales.

### Pregunta 1
¿Qué intervalo evita asignar dos versiones en el instante de cambio?

A. Ambos extremos inclusivos

B. Inicio inclusivo y fin exclusivo

C. Ambos abiertos siempre

D. Sin fecha final

### Pregunta 2
Llega tarde un evento de febrero después de uno de marzo. ¿Cómo ordenar?

A. Por orden de llegada

B. Por nombre del archivo

C. Por secuencia de negocio

D. Por ciudad alfabética

### Pregunta 3
¿Qué opción de AUTO CDC expresa el orden?

A. SEQUENCE BY / sequence_by

B. LOCATION

C. HAVING

D. OPTIMIZE

### Respuestas y explicación
**1. B** — La frontera pertenece solo a la nueva versión.

**2. C** — El modelo temporal debe respetar el orden de cambios del origen.

**3. A** — Define el orden usado al mantener la dimensión.

## PARTE 6 - RETO FINAL
Añade dos cambios consecutivos sin modificar atributos. Evita versiones redundantes y conserva intervalos correctos. Después compara el enfoque batch con resources/pipelines/auto_cdc.py del tema 22.

Anota tu decisión, implementa el código y muestra evidencias. No se incluye solución para este reto.

In [ ]:
# TU RETO: código y verificaciones
